In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.silver.orders")

daily_sales = (
    orders
    .filter(F.col("order_status").isin("COMPLETED", "SHIPPED"))
    .withColumn("order_day", F.to_date("order_date"))
    .groupBy("order_day")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("average_order_value")
    )
    .orderBy("order_day")
)

(
    daily_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.daily_sales")
)

display(daily_sales)